# Introduction

The goal of this project is to begin the legwork of a private personal investment strategy. The unique factor of this investment strategy will be the inclusion of employee reviews on "Positive Business Outlook" from Glassdoor.com. 

The constraints of this project are that the companies for consideration must be public, United States companies. The steps to complete the project are outlined throughout this notebook and include data extraction, munging, analysis, and visualization. the project also includes statistical analysis (e.g. hypothesis testing and regression).

The project should show strong skills and a solid understanding in the aforementioned topics, along with strong problem solving skills including the ability to break a problem into components and understanding how to test each component, an ability to initiate and drive projects to completion without guidance, excellent written and verbal communication skills, ability to create a meaningful presentation that tells a story, and a strong work ethic, intellectual curiosity and attention to detail.

# Problem Identification

## Part 1 - Defining Hypthoses

The first step is to define the problem. The key of this project is to determine if "Positive Business Outlook" from employee submissions on Glassdoor.com can determine the return on investment (ROI).

Thus the null hypothesis (Ho) is: Positive business outlook is not statistically correlated to 5-year return on investment. Therefore at the standard significance level of 5%, a small p-value of ≤ 0.05 indicates strong evidence against the null hypothesis, so it is rejected.

## Part 2 - Defining Dataset

The next step is to determine how much data is necessary to have sufficient data to test our theory. Since the goal is on a focus of the United States and public companies, I have choosen data requirements that can be extrapolated to that population.

### Public Companies

According to Business Insider, in 2012, there were 4,102 public companies (http://www.businessinsider.com/us-has-too-few-publicly-listed-companies-2015-6). With a 5% confidence interval/margin of error and 95% confidence level, the dataset must contain at least 352 companies.

Note: With 352 companies, it is likely that the dataset will also have an adequate sample size if it was used to sample US labor force and US dollars. With a 1% margin of error and 99% confidence level more than about 16,588 dollars or employees meets those sample size requirements.

# Data Extraction

In [ ]:
#pip install selenium

In [ ]:
#pip install webdriver_manager

In [1]:
from bs4 import BeautifulSoup
import os
import pandas as pd
import requests
import time
from selenium import webdriver
import matplotlib.pyplot as plt
from scipy import stats
import numpy as np
import seaborn as sns
import lxml
from webdriver_manager.chrome import ChromeDriverManager
from csv import writer
import random
import warnings
warnings.filterwarnings("ignore")

In [2]:
company_list = pd.read_csv("../data/interm.csv")
company_list.sort_values('ticker', inplace=True)
company_list.dropna(axis=0, how='all',inplace=True)
company_list.dropna(axis=1, how='all',inplace=True)


In [3]:
# To view initial structure of dataset
company_list.tail()

,ticker,security,outlook,rating,use,trust,upward_trend,outlook_date,sector,industry
17,PEP,Pepsi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16,PG,Proctor and Gamble,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,PYPL,PayPal Holdings,80.0,4.1,0.0,0.0,yes,2/27/21,Financial Services,Credit Services
7,SQ,Square,81.0,4.2,NaN,NaN,NaN,2/27/21,NaN,NaN
0,TSLA,Tesla,56.0,3.6,1.0,0.0,pass,11/3/20,Consumer Cyclical,Auto Manufacturers


In [4]:
company_list.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 23 entries, 22 to 0
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   ticker        23 non-null     object 
 1   security      23 non-null     object 
 2   outlook       13 non-null     float64
 3   rating        13 non-null     float64
 4   use           4 non-null      float64
 5   trust         4 non-null      float64
 6   upward_trend  4 non-null      object 
 7   outlook_date  13 non-null     object 
 8   sector        4 non-null      object 
 9   industry      4 non-null      object 
dtypes: float64(4), object(6)
memory usage: 2.0+ KB


In [5]:
def append_dict_as_row(file_name, list_of_elem):
    # Open file in append mode
    with open(file_name, 'a+', newline='') as write_obj:
        # Create a writer object from csv module
        csv_writer = writer(write_obj)
        # Add dictionary as wor in the csv
        #if count == 1:
        #    dict_writer.writerow({})
        csv_writer.writerow(list_of_elem)
    time.sleep(2)
    write_obj.close()       

In [6]:
# Web scrapping to gather company data from Yahoo Finance and Morningstar 

count = 0
total = len(company_list['ticker'])
partial = pd.read_csv("../data/partial.csv")
partial.dropna(axis=0, how='all',inplace=True)

df = partial.copy()

for ticker in company_list['ticker']:
    count+=1
    time.sleep(random.randint(2,41))
    if ticker not in partial['ticker'].values.tolist():
        sector = float('nan')
        industry = float('nan')
        employees = float('nan')

        # MORNINGSTAR
        # Profit Margin to determine MOAT (higher better), Positive EBITDA / Shares Outstanding (higher better)
        time.sleep(random.randint(2,11))
        try:
            statistics = 'http://financials.morningstar.com/ratios/r.html?t=' + ticker + '&region=usa&culture=en-US'
            driver = webdriver.Chrome(ChromeDriverManager().install())
            time.sleep(1)
            driver.get(statistics)
            time.sleep(1)
            statistics_response = driver.page_source
            time.sleep(1)
            statistics_soup = BeautifulSoup(statistics_response, 'lxml')
            time.sleep(1)
            driver.quit()
            try:
                profit_margin = float(statistics_soup.find('td', {'headers': 'pr-pro-Y10 pr-profit i22'}).contents[0])
            except (IndexError, AttributeError):
                profit_margin = float('nan')
            try:
                roe = float(statistics_soup.find('td', {'headers':'pr-pro-Y10 pr-profit i26'}).contents[0])
            except (IndexError, AttributeError):
                roe = float('nan')
            try:
                roa = float(statistics_soup.find('td', {'headers':'pr-pro-Y10 pr-profit i24'}).contents[0])
            except (IndexError, AttributeError):
                roa = float('nan')
        except:
            profit_margin = float('nan')
            roe = float('nan')
            roa = float('nan')
            
        # P/E Growth (PEG) (5 year expected) (low means under-valued), Positive EBITDA / Shares Outstanding (higher better)
        time.sleep(random.randint(2,12))
        try:
            peg_url = 'http://financials.morningstar.com/valuation/price-ratio.html?t=' + ticker + '&region=usa&culture=en-US'
            driver = webdriver.Chrome(ChromeDriverManager().install())
            time.sleep(1)
            driver.get(peg_url)
            time.sleep(1)
            peg_response = driver.page_source
            time.sleep(1)
            peg_soup = BeautifulSoup(peg_response, 'lxml')
            time.sleep(1)
            driver.quit()
            try:
                peg = float(peg_soup.find('table', {'id': 'forwardValuationTable'}).find('tr',{'scope':'row'}).find_next_sibling('tr',{'scope':'row'}).find('td').find_next_sibling('td').contents[0])
            except (IndexError, AttributeError):
                peg = float('nan')
        except:
            peg = float('nan')

        # Net Income, Research & Development
        time.sleep(random.randint(2,10))
        try:
            financials_url = 'http://financials.morningstar.com/income-statement/is.html?t=' + ticker +'&region=usa&culture=en-US'
            driver = webdriver.Chrome(ChromeDriverManager().install())
            time.sleep(1)
            driver.get(financials_url)
            time.sleep(1)
            financials_response = driver.page_source
            time.sleep(1)
            financials_soup = BeautifulSoup(financials_response, 'lxml')
            time.sleep(1)
            driver.quit()
            try:
                netincome = financials_soup.find('div', {"id": "data_i80"}).find('div', {"id": "Y_6"}).contents[0].replace(',','')
            except (IndexError, AttributeError):
                netincome = float('nan')
            try:
                rd = financials_soup.find('div', {"id": "data_i11"}).find('div', {"id": "Y_6"}).contents[0].replace(',','')
            except (IndexError, AttributeError):
                rd = float('nan')     
        except:
            netincome = float('nan')
            rd = float('nan')

        # 3, 5 and 10 Year Return
        time.sleep(random.randint(2,11))
        try:
            returns_url = 'http://performance.morningstar.com/stock/performance-return.action?t=' + ticker
            driver = webdriver.Chrome(ChromeDriverManager().install())
            time.sleep(1)
            driver.get(returns_url)
            time.sleep(1)
            returns_response = driver.page_source
            time.sleep(1)
            returns_soup = BeautifulSoup(returns_response, 'lxml')
            time.sleep(1)
            driver.quit()
            try:
                one_yr = returns_soup.find('table', class_='r_table3 width955px print97').find('th', class_='row_lbl').find_next_sibling('td', class_='row_data').find_next_sibling('td', class_='row_data').find_next_sibling('td', class_='row_data').find_next_sibling('td', class_='row_data').find_next_sibling('td', class_='row_data').find_next_sibling('td', class_='row_data').contents[0]
            except (IndexError, AttributeError):
                one_yr = float('nan')
            try:
                three_yr = returns_soup.find('table', class_='r_table3 width955px print97').find('th', class_='row_lbl').find_next_sibling('td', class_='row_data').find_next_sibling('td', class_='row_data').find_next_sibling('td', class_='row_data').find_next_sibling('td', class_='row_data').find_next_sibling('td', class_='row_data').find_next_sibling('td', class_='row_data').find_next_sibling('td', class_='row_data').contents[0]
            except (IndexError, AttributeError):
                three_yr = float('nan')
            try:
                five_yr = returns_soup.find('table', class_='r_table3 width955px print97').find('th', class_='row_lbl').find_next_sibling('td', class_='row_data').find_next_sibling('td', class_='row_data').find_next_sibling('td', class_='row_data').find_next_sibling('td', class_='row_data').find_next_sibling('td', class_='row_data').find_next_sibling('td', class_='row_data').find_next_sibling('td', class_='row_data').find_next_sibling('td', class_='row_data').contents[0]
            except (IndexError, AttributeError):
                five_yr = float('nan')
            try:
                ten_yr = returns_soup.find('table', class_='r_table3 width955px print97').find('th', class_='row_lbl').find_next_sibling('td', class_='row_data').find_next_sibling('td', class_='row_data').find_next_sibling('td', class_='row_data').find_next_sibling('td', class_='row_data').find_next_sibling('td', class_='row_data').find_next_sibling('td', class_='row_data').find_next_sibling('td', class_='row_data').find_next_sibling('td', class_='row_data').find_next_sibling('td', class_='row_data').contents[0]
            except (IndexError, AttributeError):
                ten_yr = float('nan')
            try:
                fifteen_yr = returns_soup.find('table', class_='r_table3 width955px print97').find('th', class_='row_lbl').find_next_sibling('td', class_='row_data').find_next_sibling('td', class_='row_data').find_next_sibling('td', class_='row_data').find_next_sibling('td', class_='row_data').find_next_sibling('td', class_='row_data').find_next_sibling('td', class_='row_data').find_next_sibling('td', class_='row_data').find_next_sibling('td', class_='row_data').find_next_sibling('td', class_='row_data').find_next_sibling('td', class_='row_data_0').contents[0]
            except (IndexError, AttributeError):
                fifteen_yr = float('nan')
            try:
                div_yield = returns_soup.find('table', class_='r_table3 print97').find('tr', class_='action').find_next_sibling('tr', class_='action').find_next_sibling('tr', class_='action').find_next_sibling('tr', class_='action').find_next_sibling('tr', class_='action').find_next_sibling('tr', class_='action').find('td', class_='row_data_0 divide').contents[0]
            except (IndexError, AttributeError):
                div_yield = float('nan')
        except:
            one_yr = float('nan')
            three_yr = float('nan')
            five_yr = float('nan')
            ten_yr = float('nan')
            fifteen_yr = float('nan')
            div_yield = float('nan')

        field_names = ['ticker','sector','industry','webscrape_date','net_income','ft_employee',
                      'div_yield','one_year','three_year','five_year','ten_year','fifteen_year','rd','peg','profit_margin',
                      'roa','roe']
        row_dict = [ticker, sector, industry, time.strftime("%x"),
                    netincome, employees, div_yield, one_yr, three_yr, 
                    five_yr, ten_yr, fifteen_yr, rd, peg, profit_margin,
                    roa, roe]

        # Append a dict as a row in csv file
        append_dict_as_row('../data/partial.csv', row_dict)
        df = df.append(row_dict, ignore_index=True)
 
        if round(count*100/total,0) == 50 or round(count*100/total,0) == 25 or round(count*100/total,0) == 75:
            print("Completed: ", ticker, "Percent Complete: ", round(count*100/total,0))
            
            

[WDM] - Current google-chrome version is 89.0.4389
[WDM] - Get LATEST driver version for 89.0.4389
[WDM] - There is no [mac64] chromedriver for browser 89.0.4389 in cache
[WDM] - Get LATEST driver version for 89.0.4389
[WDM] - Trying to download new driver from http://chromedriver.storage.googleapis.com/89.0.4389.23/chromedriver_mac64.zip


[WDM] - Driver has been saved in cache [/Users/kchebs/.wdm/drivers/chromedriver/mac64/89.0.4389.23]
[WDM] - Current google-chrome version is 89.0.4389
[WDM] - Get LATEST driver version for 89.0.4389
[WDM] - Driver [/Users/kchebs/.wdm/drivers/chromedriver/mac64/89.0.4389.23/chromedriver] found in cache


[WDM] - Current google-chrome version is 89.0.4389
[WDM] - Get LATEST driver version for 89.0.4389
[WDM] - Driver [/Users/kchebs/.wdm/drivers/chromedriver/mac64/89.0.4389.23/chromedriver] found in cache


[WDM] - Current google-chrome version is 89.0.4389
[WDM] - Get LATEST driver version for 89.0.4389
[WDM] - Driver [/Users/kchebs/.wdm/drivers/chromedriver/mac64/89.0.4389.23/chromedriver] found in cache


[WDM] - Current google-chrome version is 89.0.4389
[WDM] - Get LATEST driver version for 89.0.4389
[WDM] - Driver [/Users/kchebs/.wdm/drivers/chromedriver/mac64/89.0.4389.23/chromedriver] found in cache


...

[WDM] - Current google-chrome version is 89.0.4389
[WDM] - Get LATEST driver version for 89.0.4389
[WDM] - Driver [/Users/kchebs/.wdm/drivers/chromedriver/mac64/89.0.4389.23/chromedriver] found in cache


[WDM] - Current google-chrome version is 89.0.4389
[WDM] - Get LATEST driver version for 89.0.4389
[WDM] - Driver [/Users/kchebs/.wdm/drivers/chromedriver/mac64/89.0.4389.23/chromedriver] found in cache


[WDM] - Current google-chrome version is 89.0.4389
[WDM] - Get LATEST driver version for 89.0.4389
[WDM] - Driver [/Users/kchebs/.wdm/drivers/chromedriver/mac64/89.0.4389.23/chromedriver] found in cache


[WDM] - Current google-chrome version is 89.0.4389
[WDM] - Get LATEST driver version for 89.0.4389
[WDM] - Driver [/Users/kchebs/.wdm/drivers/chromedriver/mac64/89.0.4389.23/chromedriver] found in cache


[WDM] - Current google-chrome version is 89.0.4389
[WDM] - Get LATEST driver version for 89.0.4389
[WDM] - Driver [/Users/kchebs/.wdm/drivers/chromedriver/mac64/89.0.4389.23/chromedriver] found in cache


In [7]:
df

,0,div_yield,fifteen_year,five_year,ft_employee,net_income,one_year,peg,profit_margin,rd,roa,roe,ten_year,three_year,ticker,webscrape_date
0,NaN,—,—,—,NaN,-102.0,324.74,28.2,-13.40,187,-6.79,-13.33,—,—,CRWD,1/1/21
1,NaN,—,22.66,34.93,NaN,279.0,54.48,3.1,17.57,158,5.31,6.48,32,46.01,CSGP,1/1/21
2,NaN,—,-7.98,-16.5,NaN,-173.0,156.21,0.0,NaN,—,NaN,NaN,-11.5,20.37,CYH,1/1/21
3,NaN,—,—,—,NaN,-283.0,—,0.0,NaN,146,NaN,NaN,—,—,DASH,1/1/21
4,NaN,—,—,—,NaN,-7.0,160.56,0.0,-1.39,179,-0.53,-0.88,—,—,DDOG,1/1/21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
481,1491,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
482,4.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
483,2.19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
484,1.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
        field_names = ['ticker','sector','industry','webscrape_date','net_income','ft_employee',
                      'div_yield','one_year','three_year','five_year','ten_year','fifteen_year','rd','peg','profit_margin',
                      'roa','roe']
        row_dict = {'ticker': ticker,
                            'sector': sector,
                            'industry': industry,
                            'webscrape_date': time.strftime("%x"),
                            'net_income': netincome,
                            'ft_employee': employees,
                            'div_yield': div_yield,
                            'one_year': one_yr,
                            'three_year': three_yr,
                            'five_year': five_yr,
                            'ten_year': ten_yr,
                            'fifteen_year': fifteen_yr,
                            'rd': rd,
                            'peg': peg,
                            'profit_margin': profit_margin,
                            'roa': roa,
                            'roe': roe}

In [8]:
# Merging new and old dataset
merged = pd.merge(company_list, partial, on='ticker', how='left')


In [9]:
# Saving dataset offline to save time so that re-running previous code 
# will not be required

merged.to_csv('../data/2020_12_v2.csv')

# Data Munging / Wrangling 

First, I manually looked through the data for fun and filled in some of the "alcohol, adult, gambling, tobacco, controversial weapons, small arms, military, and coal" categories for companies I knew but Yahoo Finance still had to create reports for. These categories state Yes or No to if the company works in those categories.

In [ ]:
# Imported New File
post_manual = pd.read_csv("../data/2020_09.csv")

In [ ]:
# Fill in blanks
merged["ft_employee"].fillna(merged["employees"], inplace=True)
merged["sector_y"].fillna(merged["sector"], inplace=True)
merged["industry_y"].fillna(merged["industry"], inplace=True)

# Removing Duplicate Column
merged.drop(['employees','sector','industry'], 1, inplace=True)

In [ ]:
merged.info()

In [ ]:
merged.drop(['Unnamed: 23','Unnamed: 24','Unnamed: 25'], 1, inplace=True)

In [ ]:
post_manual.info()

In [ ]:
post_manual['net_income'].replace(",","", inplace = True)
post_manual['ft_employee'].replace(",","", inplace = True)

In [ ]:
for i in range(len(post_manual)):
    post_manual['ft_employee'][i] = str(post_manual['ft_employee'][i]).replace(",","")

In [ ]:
post_manual['net_income'] = post_manual['net_income'].astype('float')
post_manual['ft_employee'] = post_manual['ft_employee'].astype('float')

In [ ]:
# Calculate Net Income Per an Employee

post_manual['net_per_emp'] = post_manual.apply(lambda row: row['net_income'] / row['ft_employee'], axis = 1)
post_manual['ebitda_share'] = post_manual.apply(lambda row: row['ebitda'] / row['shares'], axis = 1)

# Check / Test
post_manual.head()

In [ ]:
# Notice how the number of non-null values have increased
post_manual.info()

In [ ]:
# Format Changes

#post_manual.sector_y = post_manual.sector_y.astype('category')
#post_manual.industry_y = post_manual.industry_y.astype('category')

post_manual['alcohol'] = post_manual['alcohol'].map({'yes': 1, 'no': 0})
post_manual['adult'] = post_manual['adult'].map({'yes': 1, 'no': 0})
post_manual['gambling'] = post_manual['gambling'].map({'yes': 1, 'no': 0})
post_manual['tobacco'] = post_manual['tobacco'].map({'yes': 1, 'no': 0})
post_manual['controversial_weapons'] = post_manual['controversial_weapons'].map({'yes': 1, 'mo': 0})
post_manual['small_arms'] = post_manual['small_arms'].map({'yes': 1, 'no': 0})
post_manual['military_contracting'] = post_manual['military_contracting'].map({'yes': 1, 'no': 0})
post_manual['coal'] = post_manual['coal'].map({'yes': 1, 'no': 0})
post_manual['use'] = post_manual['use'].map({'yes': 1, 'no': 0})
post_manual['trust'] = post_manual['trust'].map({'yes': 1, 'no': 0})



In [ ]:
post_manual.to_csv('../data/2020_09.csv')

# Analysis

In [ ]:
calcdf = pd.read_csv("second_merge_aug2019.csv")

calcdf.info()

In [ ]:
calcdf = calcdf.drop('Unnamed: 0', 1)

calcdf.describe()

In [ ]:
# Correlation and P-Values

matrix = calcdf
matrix = matrix.drop('date',1)
matrix = matrix.drop('ticker',1)
matrix = matrix.drop('security',1)
matrix = matrix.drop('sector',1)
matrix = matrix.drop('industry',1)
matrix = matrix.drop('adult',1)

rho, pval = stats.spearmanr(matrix)
# print('stats.spearmanr - cor:\n', rho)
# print('stats.spearmanr - pval\n', pval)

#pval

# round(pd.DataFrame(pval), 3)

pvals = pd.DataFrame(pval, columns=['outlook','net_income','ft_employee','div_yield','five_year','ten_year','alcohol','gambling','tobacco','controversial_weapons','small_arms','military_contracting','coal','dont_use','dont_trust','assets','market_value','revenue','net_per_emp'])
pvals = pvals.rename(index={0: 'outlook',1: 'net_income',2: 'ft_employee',3: 'div_yield',4: 'five_year',5: 'ten_year',6: 'alcohol',7: 'gambling',8: 'tobacco',9: 'controversial_weapons',10: 'small_arms',11: 'military_contracting',12: 'coal',13: 'dont_use',14: 'dont_trust',15: 'assets',16: 'market_value',17: 'revenue',18: 'net_per_emp'})
round(pvals, 3)

In [ ]:
mask = np.zeros_like(pvals)
mask[np.triu_indices_from(mask)] = True
with sns.axes_style("white"):
    p2 = sns.heatmap(pvals, mask=mask, square=True, cmap="PuOr")

In [ ]:
# Not normally distributed so Spearman
cm = matrix.corr(method='spearman')
cm

In [ ]:
sns.heatmap(cm, square=True)
plt.yticks(rotation=0)
plt.xticks(rotation=90)

In [ ]:
items = ['outlook','net_income','ft_employee','div_yield','five_year','ten_year','alcohol','gambling','tobacco','controversial_weapons','small_arms','military_contracting','coal','dont_use','dont_trust','assets','market_value','revenue','net_per_emp']
arr = []

for i in items:
    for j in items:
        if pvals.loc[i,j] <= 0.05 and (cm.loc[i,j] > 0.5 or cm.loc[i,j] < -0.5) and i != j and [j,i, pvals.loc[j,i], cm.loc[j,i]] not in arr:
            arr.append([i,j, pvals.loc[i,j], cm.loc[i,j]])
        

arr = pd.DataFrame(arr,columns=['1','2','P-Value','R'])
arr.sort_values('R', ascending = True)

In [ ]:
five_year_df = calcdf[calcdf.outlook.notnull()]
five_year_df = five_year_df[five_year_df.five_year.notnull()]
five_year_df

x, y = five_year_df.outlook, five_year_df.five_year
slope, intercept, r_value, p_value, std_err = stats.linregress(x,y)
line = x*slope + intercept
five_corr = plt.plot(x,y, 'x', x, line, color = 'black')
plt.xlabel('% Positive Business Outlook')
plt.ylabel('% Return on Investment')
plt.show()

print('Line Equation: Five Year Return = ({})*Outlook +{}'.format(slope,intercept))
print('Line of Best Fit Correlation: {}'.format(r_value))

In [ ]:
ten_year_df = calcdf[calcdf.outlook.notnull()]
ten_year_df = ten_year_df[ten_year_df.ten_year.notnull()]
ten_year_df

x, y = ten_year_df.outlook, ten_year_df.five_year
slope, intercept, r_value, p_value, std_err = stats.linregress(x,y)
line = x*slope + intercept
five_corr = plt.plot(x,y, 'x', x, line, color = 'black')
plt.xlabel('% Positive Business Outlook')
plt.ylabel('% Return on Investment')
plt.show()

print('Line Equation: Ten Year Return = ({})*Outlook +{}'.format(slope,intercept))
print('Line of Best Fit Correlation: {}'.format(r_value))

In future data collections, the information will be gathered but the date will change. Thus, it will be possible to develop a time series forcast.

In [ ]:
X = calcdf[np.isfinite(calcdf['outlook'])]
X = X[np.isfinite(X['net_income'])]
X = X[np.isfinite(X['ft_employee'])]
X = X[np.isfinite(X['div_yield'])]
X = X[np.isfinite(X['five_year'])]
X = X[np.isfinite(X['assets'])]
X = X[np.isfinite(X['market_value'])]
X = X[np.isfinite(X['revenue'])]

Y = np.array(X['five_year'])
X = np.array(X.drop(['ticker','security','sector','industry','alcohol','adult','gambling','tobacco','controversial_weapons',
                          'small_arms','military_contracting','coal','dont_use','dont_trust','five_year','ten_year'],1))

In [ ]:
from sklearn import preprocessing, cross_validation
from sklearn.linear_model import LinearRegression

In [ ]:
X = preprocessing.scale(X)

In [ ]:
X

In [ ]:
# Checking Lengths Match
print(len(X), len(Y))

X_train, X_test, Y_train, Y_test = cross_validation.train_test_split(X, Y, test_size=0.2)

clf = LinearRegression(n_jobs = -1)
clf.fit(X_train, Y_train)

squarederror = clf.score(X_test, Y_test)

print(squarederror)

# Predicted
forecast = clf.predict(X)

print(forecast)

## Investment Rules
- Net Income per Employee >= 10,000 (Kevin)
- Profit Margin > 2.57% (Warren Buffett)
- 3, 5, and 10 year ROI > 10.1%
- Upward Trend (Kevin)
- Business Outlook >= 60 (Kevin)
    - Employees are confident in future of company
- Rating >= 3.5 (Kevin)
    - Employees have positive things to say about company
- PEG >= 0 and <= 1 (Warren Buffett)
    - Thus, the stock is ranked as under-valued which is desirable because it means it has a lot of room for growth
- Beats 1 and 5 year SP500